# WandB Run Comparison by ID

Fetches specific experiment runs from WandB and compares metrics, heatmaps, and ridge plots side by side.

Each run pair (DPG + DiCE for a given dataset) gets:
- **Metrics table** — key comparison metrics from both techniques
- **Heatmap** — `heatmap_techniques` showing feature-level CF differences
- **Ridge plot** — `plot_ridge_comparison` showing CF distributions vs dataset

In [ ]:
# Configuration — WandB run IDs to compare
RUN_IDS = ['bgyxnb4h', 'zkljo11n', 'srjd3c3b', 'lpkj67ca', 'yv2ab6ib']

ENTITY = 'mllab-ts-universit-di-trieste'
PROJECT = 'CounterFactualDPG'

In [ ]:
import sys
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Add counterfactual root to path
_cf_root = os.path.abspath('..')
_repo_root = os.path.abspath(os.path.join('..', '..'))
for p in [_cf_root, _repo_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import wandb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from counterfactual_visualizer import heatmap_techniques, plot_ridge_comparison
from utils.compare_techniques import COMPARISON_METRICS
from utils.dataset_loader import load_dataset
from utils.config_manager import load_config

print('Imports OK')

## Fetch runs from WandB

In [ ]:
api = wandb.Api(timeout=60)

runs_info = {}  # run_id -> {meta, config, summary, metrics}

for run_id in RUN_IDS:
    run = api.run(f'{ENTITY}/{PROJECT}/{run_id}')
    config = run.config
    summary = run.summary._json_dict

    data_cfg = config.get('data', {})
    dataset_name = data_cfg.get('dataset_name') or data_cfg.get('dataset', 'unknown')

    # Infer technique
    technique = data_cfg.get('method', '')
    if not technique:
        name_lower = run.name.lower()
        if '_dpg' in name_lower or name_lower.endswith('dpg'):
            technique = 'dpg'
        elif '_dice' in name_lower or name_lower.endswith('dice'):
            technique = 'dice'

    # Extract comparison metrics
    metrics = {}
    for metric_key, metric_info in COMPARISON_METRICS.items():
        for wkey in metric_info.get('wandb_keys', [f'metrics/{metric_key}']):
            if wkey in summary:
                metrics[metric_key] = summary[wkey]
                break

    # Extract sample, counterfactuals, constraints
    feature_names = config.get('feature_names', [])

    sample = config.get('sample') or summary.get('sample') or summary.get('original_sample')
    if isinstance(sample, list) and feature_names:
        sample = dict(zip(feature_names, sample))

    sample_class = config.get('sample_class')
    if sample_class is None:
        sample_class = summary.get('sample_class') or summary.get('original_class')

    cfs = summary.get('final_counterfactuals', [])
    if isinstance(cfs, str):
        try:
            cfs = json.loads(cfs)
        except Exception:
            cfs = []
    if cfs and isinstance(cfs[0], list) and feature_names:
        cfs = [dict(zip(feature_names, cf)) for cf in cfs]

    restrictions = config.get('restrictions')
    constraints = config.get('dpg', {}).get('constraints') if config.get('dpg') else None

    runs_info[run_id] = {
        'name': run.name,
        'state': run.state,
        'dataset': dataset_name,
        'technique': technique.lower() if technique else 'unknown',
        'metrics': metrics,
        'sample': sample,
        'sample_class': sample_class,
        'counterfactuals': cfs,
        'restrictions': restrictions,
        'constraints': constraints,
        'feature_names': feature_names,
    }
    print(f'  {run_id}  {run.name}  dataset={dataset_name}  technique={technique}  '
          f'state={run.state}  #cfs={len(cfs)}  #metrics={len(metrics)}')

print(f'\nFetched {len(runs_info)} runs.')

## Group runs by dataset (DPG + DiCE pairs)

In [ ]:
from collections import defaultdict

# Group by dataset → technique → run_id
dataset_pairs = defaultdict(dict)  # {dataset: {technique: run_id}}

for run_id, info in runs_info.items():
    ds = info['dataset']
    tech = info['technique']
    dataset_pairs[ds][tech] = run_id

for ds, techs in dataset_pairs.items():
    print(f'{ds}: {techs}')

## Metrics comparison table

In [ ]:
KEY_METRICS = [
    'perc_valid_cf_all', 'perc_actionable_cf_all',
    'plausibility_nbr_cf', 'distance_mh',
    'avg_nbr_changes', 'count_diversity_all',
    'accuracy_knn_sklearn', 'runtime',
]

for ds, techs in sorted(dataset_pairs.items()):
    display(Markdown(f'### {ds}'))

    rows = []
    for tech in ['dpg', 'dice']:
        rid = techs.get(tech)
        if not rid:
            continue
        m = runs_info[rid]['metrics']
        row = {'technique': tech.upper(), 'run_id': rid}
        for k in KEY_METRICS:
            val = m.get(k)
            if val is not None:
                fmt = COMPARISON_METRICS.get(k, {}).get('format', '.4f')
                try:
                    row[k] = format(val, fmt)
                except (ValueError, TypeError):
                    row[k] = val
            else:
                row[k] = '-'
        rows.append(row)

    if rows:
        display(pd.DataFrame(rows).set_index('technique'))

## Heatmap & Ridge visualizations per dataset

For each dataset that has both a DPG and a DiCE run, generate:
1. `heatmap_techniques` — feature-level comparison of counterfactuals
2. `plot_ridge_comparison` — distribution-based comparison with the original dataset

In [ ]:
# Helper: load dataset and train a model (for ridge plot)
_dataset_cache = {}

def get_dataset_model(dataset_name):
    """Load dataset, train RF model, return (dataset_df, target, model)."""
    if dataset_name in _dataset_cache:
        return _dataset_cache[dataset_name]

    config_path = os.path.join(_cf_root, 'configs', dataset_name, 'config.yaml')
    if not os.path.exists(config_path):
        print(f'  Config not found: {config_path}')
        return None

    config = load_config(config_path, repo_root=_cf_root)
    seed = getattr(config.experiment_params, 'seed', 42)
    np.random.seed(seed)

    ds = load_dataset(config, repo_root=_cf_root)
    features_df = ds['features_df']
    labels = ds['labels']

    X_train, X_test, y_train, y_test = train_test_split(
        features_df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)

    result = {
        'dataset_df': features_df,
        'target': labels,
        'model': model,
    }
    _dataset_cache[dataset_name] = result
    return result

print('Helper ready')

In [ ]:
for ds, techs in sorted(dataset_pairs.items()):
    dpg_id = techs.get('dpg')
    dice_id = techs.get('dice')

    if not dpg_id or not dice_id:
        print(f'⚠ {ds}: missing DPG or DiCE run, skipping visualizations')
        continue

    dpg_info = runs_info[dpg_id]
    dice_info = runs_info[dice_id]

    sample = dpg_info['sample']
    sample_class = dpg_info['sample_class']
    dpg_cfs = dpg_info['counterfactuals']
    dice_cfs = dice_info['counterfactuals']
    restrictions = dpg_info['restrictions']
    constraints = dpg_info['constraints']

    if not sample or not dpg_cfs or not dice_cfs:
        print(f'⚠ {ds}: missing sample or counterfactuals')
        continue

    display(Markdown(f'---\n## {ds}'))
    display(Markdown(f'DPG run `{dpg_id}` — DiCE run `{dice_id}`'))

    # --- Heatmap ---
    display(Markdown('### Heatmap — technique comparison'))
    fig_hm = heatmap_techniques(
        sample=sample,
        class_sample=sample_class,
        cf_list_1=dpg_cfs[:5],
        cf_list_2=dice_cfs[:5],
        technique_names=('DPG', 'DiCE'),
        restrictions=restrictions,
    )
    if fig_hm:
        display(fig_hm)
        plt.close(fig_hm)

    # --- Ridge plot ---
    dm = get_dataset_model(ds)
    if dm is None:
        print(f'  ⚠ Could not load dataset/model for ridge plot')
        continue

    dataset_df = dm['dataset_df']
    target = dm['target']
    model = dm['model']

    sample_df = pd.DataFrame([sample])
    original_class = model.predict(sample_df)[0]
    target_class = model.predict(pd.DataFrame([dpg_cfs[0]]))[0] if dpg_cfs else None

    display(Markdown('### Ridge — distribution comparison'))
    fig_ridge = plot_ridge_comparison(
        sample=sample,
        cf_list_1=dpg_cfs,
        cf_list_2=dice_cfs,
        technique_names=('DPG', 'DiCE'),
        method_1_color='#CC0000',
        method_2_color='#006DAC',
        dataset_df=dataset_df,
        constraints=constraints,
        target=target,
        target_class=target_class,
        original_class=original_class,
        show_per_class_distribution=True,
        show_overall_distribution=False,
        show_original_class_constraints=False,
    )
    if fig_ridge:
        display(fig_ridge)
        plt.close(fig_ridge)